## Script atualizado para trabalhar com o XML Novo

In [ ]:
from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

In [ ]:
# Função para carregar os arquivos
base_path = r"C:\Dumps\todos_enms"

def load_parquet(pattern, columns=None):
    import glob, pandas as pd
    files = glob.glob(pattern)
    if not files:
        return pd.DataFrame()
    df = pd.concat((pd.read_parquet(f, columns=columns) for f in files),
                   ignore_index=True)
    try:
        df = df.convert_dtypes(dtype_backend="pyarrow")
    except Exception:
        pass
    return df


In [ ]:
import pandas as pd
import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\EUtranCellFDD_*_part*.parquet")

# Adiciona as colunas se não existirem
if 'eUtranCellFDDId' not in df1.columns:
    df1['eUtranCellFDDId'] = np.nan
if 'EUtranCellFDDId' not in df1.columns:
    df1['EUtranCellFDDId'] = np.nan

# Agora pode usar combine_first com segurança
df1['CELL'] = df1['eUtranCellFDDId'].combine_first(df1['EUtranCellFDDId'])

# df1.rename(columns={'eUtranCellFDDId': 'CELL'},inplace=True) 

#df['UF']= df['eNB'].str[-2:].isin(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE']) # selecionar as UFs desejadas
df1['PCI']= df1['physicalLayerCellIdGroup'].astype("Int64")*3 + df1['physicalLayerSubCellId'].astype("Int64") # calcular o PCI usando as colunas existentes

df1.dropna(subset=['earfcndl'], inplace=True) # comando para deletar as linhas NULL da coluna escolhida

# df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['eNB'] = df1['ManagedElement']
df1['earfcndl']= df1['earfcndl'].astype(int)

df1['PCI']= df1['PCI'].astype(int)

df1['UF']= df1['eNB'].str[-2:]

df2= pd.DataFrame(df1)
# df2.drop('eUtranCellFDDId', axis='columns', inplace=True)

df3= df2.dropna(thresh=1, axis='columns')


df3.to_csv('C:/Dumps/LTE_total/EUtranCellFDD.txt', sep='\t', index=None)
print('EUtranCellFDD OK')

In [ ]:
import pandas as pd
import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ENodeBFunction_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1['UF']= df1['eNB'].str[-2:]

df3= df1.dropna(thresh=1, axis='columns')

df3.to_csv('C:/Dumps/LTE_total/ENodeBFunction.txt', sep='\t', index=None)
print('ENodeBFunction OK')

In [ ]:
import pandas as pd
import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\UtranFreqRelation_*_part*.parquet")

# Com o comando acima o eNB viria com "MeContext="" antes do nome da eNB, para deixar apenas o eNB name, usar o comando abaixo
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1['UF']= df1['eNB'].str[-2:]

# Renomeando a coluna CELL
df1.rename(columns={'EUtranCell': 'CELL'},inplace=True)

pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth', None)

df1['FREQ_UMTS'] = df1['utranFreqRelationId'].astype(str).str.rsplit('_', n=1).str[-1]

df1['UtranFreqRelationId']= df1['UtranFreqRelationId'].fillna('')+df1['utranFreqRelationId'].fillna('')

df2= pd.DataFrame(df1)
df2.drop('utranFreqRelationId', axis='columns', inplace=True) 

df2= df2.dropna(thresh=1, axis='columns')

df2.to_csv('C:/Dumps/LTE_Total/UtranFreqRelation.txt', sep='\t', index=None)
print('UtranFreqRelation OK')

In [ ]:
import pandas as pd
import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\RetSubUnit_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1['UF']= df1['eNB'].str[-2:]

df1['FREQ'] = df1['userLabel'].astype(str).str.rsplit('_', n=1).str[-1]

df2= pd.DataFrame(df1)
df2.drop(['RetSubUnitId', 'retSubUnitId'], axis='columns', inplace=True) 
df3= df2.dropna(thresh=1, axis='columns')

df3.to_csv('C:/Dumps/LTE_Total/RetSubUnit.txt', sep='\t', index=None)
df3.to_csv('C:/Vivo Reports/LTE/LTE MOs/RetSubUnit.csv', encoding='ISO-8859-1', index=None)
print('RetSubUnit OK')

In [ ]:
# Essa forma de concatenar o EUtranCellRelation funcionou lindo!!!
# Versão atualizada (VSCode / ambientes diferentes / parquet seguro)

import pandas as pd
import glob
import gc

# --------------------------------------------------
# Detecta automaticamente o engine parquet disponível
# --------------------------------------------------
def get_parquet_engine():
    try:
        import pyarrow  # noqa
        print("Usando engine: pyarrow")
        return "pyarrow"
    except ImportError:
        try:
            import fastparquet  # noqa
            print("Usando engine: fastparquet")
            return "fastparquet"
        except ImportError:
            raise ImportError(
                "Nenhum engine parquet instalado.\n"
                "Execute: pip install pyarrow"
            )

# --------------------------------------------------

def solution_concatenate_parquet():

    engine = get_parquet_engine()

    files = glob.glob(fr"{base_path}\EUtranCellRelation_*_part*.parquet")

    arquivo_txt = r'C:/Dumps/LTE_Total/EUtranCellRelation.txt'
    arquivo_vivo = r'C:/Vivo Reports/MoB_IMP/EUtranCellRelation.txt'

    if not files:
        print("Nenhum arquivo parquet encontrado.")
        return

    print(f"Carregando {len(files)} arquivos...")

    dfs = []

    for i, file in enumerate(files):
        print(f"Lendo {i+1}/{len(files)}: {file}")
        df = pd.read_parquet(file, engine=engine)
        dfs.append(df)

    # --------------------------------------------------
    # CONCATENAÇÃO
    # --------------------------------------------------
    print("Concatenando...")
    df1 = pd.concat(dfs, ignore_index=True, sort=False)

    # libera memória
    del dfs
    gc.collect()

    # --------------------------------------------------
    # PROCESSAMENTO
    # --------------------------------------------------
    print("Processando dados...")

    df1 = df1[df1['EUtranCell'].notna()]

    if 'eUtranCellRelationId' in df1.columns:
        df1['EUtranCellRelationId'] = (
            df1['EUtranCellRelationId']
            .combine_first(df1['eUtranCellRelationId'])
        )
        df1.drop(columns='eUtranCellRelationId', inplace=True)

    df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

    df1 = df1.dropna(thresh=1, axis='columns')

    if 'EUtranFreqRelation' in df1.columns:
        df1['Freq_tgt'] = df1['EUtranFreqRelation'].str.replace('LTE_', '', regex=False)

    if 'eNB' in df1.columns:
        df1['UF'] = df1['eNB'].str[-2:]

    # --------------------------------------------------
    # REORGANIZA COLUNAS
    # --------------------------------------------------
    priority_cols = ['MO', 'UF', 'eNB', 'CELL', 'EUtranCellRelationId', 'Freq_tgt']

    existing_priority = [c for c in priority_cols if c in df1.columns]
    other_cols = [c for c in df1.columns if c not in existing_priority]

    df1 = df1[existing_priority + other_cols]

    # --------------------------------------------------
    # LIMPEZA FINAL
    # --------------------------------------------------
    df1.drop_duplicates(inplace=True)

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------
    print("Salvando arquivos...")
    df1.to_csv(arquivo_txt, sep='\t', index=False)
    df1.to_csv(arquivo_vivo, sep='\t', index=False)

    print(f'EUtranCellRelation OK - {len(df1):,} linhas processadas')


# --------------------------------------------------
# DEFINA O CAMINHO
# --------------------------------------------------
# base_path = r"C:/seu/caminho/aqui"

solution_concatenate_parquet()

# Atualização do EUtranFreqRelation separando os dados do 4G e 5G

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\EUtranFreqRelation_*_part*.parquet")

df1 = df1[~df1['EUtranCell'].isna()]
df1.rename(columns={'EUtranCell': 'CELL'},inplace=True)
df1['Tecnologia']='LTE'
df1= df1.dropna(thresh=1, axis='columns')
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]
df1['EUtranFreqRelationId']= df1['EUtranFreqRelationId'].fillna('')+df1['eUtranFreqRelationId'].fillna('')
df1['earfcndl']= df1['EUtranFreqRelationId']
df2= pd.DataFrame(df1)

df2.drop('eUtranFreqRelationId', axis='columns', inplace=True)
df2.dropna(subset=['connectedModeMobilityPrio'], inplace=True)

df2.to_csv('C:/Dumps/LTE_Total/EUtranFreqRelation.txt', sep='\t', index=None)

print('EUtranFreqRelation OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\TermPointToMme_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['Termpoint']= df1['TermPointToMmeId'].fillna('')+df1['termPointToMmeId'].fillna('')

df1.drop('termPointToMmeId', axis='columns', inplace=True)

df2= df1.dropna(thresh=100, axis='columns')

df2 = df2.copy()

df2['Regional']= df2['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N','N','N','N','N', 'BASE', 'BASE'])

df3= df2.set_index('Regional')

df3.drop_duplicates(subset=['eNB', 'Termpoint', 'administrativeState'],inplace=True)

df3.to_csv('C:/Dumps/LTE_Total/TermPointToMme.txt', sep='\t')
df3.to_csv('C:/Vivo Reports/LTE/LTE MOs/TermPointToMme.csv')
print('TermPointToMme OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigSearch_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1.rename(columns={'EUtranCell': 'CELL'},inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1.dropna(subset=['CELL'], inplace=True)
df1= df1.dropna(thresh=100, axis='columns')
df1.to_csv('C:/Dumps/LTE_Total/ReportConfigSearch.txt', sep='\t', index=None)

print('ReportConfigSearch OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigEUtraInterFreqLb_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]
df1.rename(columns={'EUtranCell': 'CELL'},inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])


df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigEUtraInterFreqLb.txt', sep='\t', index=None)

print('ReportConfigEUtraInterFreqLb OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigA5_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1.rename(columns={'EUtranCell': 'CELL'},inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigA5.txt', sep='\t', index=None)

print('ReportConfigA5 OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigA5UlTrig_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1.rename(columns={'EUtranCell': 'CELL'},inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigA5UlTrig.txt', sep='\t', index=None)

print('ReportConfigA5UlTrig OK')                                        

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\SectorCarrier_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['SectorCarrierId']= df1['SectorCarrierId'].fillna('')+df1['sectorCarrierId'].fillna('')
df1['CELL']= df1['SectorCarrierId']

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df2= pd.DataFrame(df1)
df2.drop('sectorCarrierId', axis='columns', inplace=True)
df3= df2.dropna(thresh=100, axis='columns')

df3.to_csv('C:/Dumps/LTE_Total/SectorCarrier.txt', sep='\t', index=None)
df3.to_csv('C:/Vivo Reports/LTE/LTE MOs/SectorCarrier.csv', encoding='ISO-8859-1', index=None)
print('SectorCarrier OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\loadbalancingFunction_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1.drop(['loadBalancingFunctionId', 'LoadBalancingFunctionId'], axis='columns', inplace=True)

df2= pd.DataFrame(df1)

df3= df2.dropna(thresh=1, axis='columns')

df3.to_csv('C:/Dumps/LTE_Total/loadbalancingFunction.txt', sep='\t', index=None)
print('loadbalancingFunction OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\QciProfilePredefined_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['QciProfilePredefinedId']= df1['qciProfilePredefinedId'].fillna('')+df1['QciProfilePredefinedId'].fillna('')

df2= pd.DataFrame(df1)

df2.drop('qciProfilePredefinedId', axis='columns', inplace=True)

df3= df2.dropna(thresh=1, axis='columns')

df3.to_csv('C:/Dumps/LTE_Total/QciProfilePredefined.txt', sep='\t', index=None)
print('QciProfilePredefined OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\GigaBitEthernet_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df2= df1.dropna(thresh=1, axis='columns')

df2.to_csv('C:/Dumps/LTE_Total/GigaBitEthernet.txt', sep='\t', index=None)

print('GigaBitEthernet OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\EthernetPort_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]
df2= df1.dropna(thresh=1, axis='columns')

df3= df2.loc[df2['eNB'].str[:1].isin(['T', 'M']), :]

df3.to_csv('C:/Dumps/LTE_Total/EthernetPort.txt', sep='\t', index=None)

print('EthernetPort OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\FeatureState_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df2= df1.dropna(thresh=1, axis='columns')

df3= df2.loc[df1['eNB'].str[:1].isin(['T', 'M']),:]

df3.to_csv('C:/Dumps/LTE_Total/FeatureState.txt', sep='\t', index=None)

print('FeatureState OK')   

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\OptionalFeatureLicense_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df2= df1.dropna(thresh=1, axis='columns')

df3= df2.loc[df1['eNB'].str[:1].isin(['T', 'M']),:]

df3.to_csv('C:/Dumps/LTE_Total/OptionalFeatureLicense.txt', sep='\t', index=None)

print('OptionalFeatureLicense OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigB2Utra_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigB2Utra.txt', sep='\t', index=None)

print('ReportConfigB2Utra OK')

In [ ]:
import pandas as pd
from glob import glob
import csv

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\DataRadioBearer_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['DataRadioBearerId']= df1['DataRadioBearerId'].fillna('').astype(str)+df1['dataRadioBearerId'].fillna('').astype(str)

df1.drop('dataRadioBearerId', axis='columns', inplace=True)

df1= df1[df1['GNBDUFunction'].isna()]
df2= df1.dropna(thresh=1, axis='columns')
df2= df2[df2['eNB'].str[:1]!='W']

df2.to_csv('C:/Dumps/LTE_Total/DataRadioBearer.txt', sep='\t', index=None)
print('DataRadioBearer OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\SignalingRadioBearer_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['SignalingRadioBearerId']= df1['SignalingRadioBearerId'].fillna('').astype(str)+df1['signalingRadioBearerId'].fillna('').astype(str)

df1.drop('signalingRadioBearerId', axis='columns', inplace=True)

df1= df1[df1['GNBDUFunction'].isna()]
df2= df1.dropna(thresh=1, axis='columns')
df2= df2[df2['eNB'].str[:1]!='W']

df3.to_csv('C:/Dumps/LTE_Total/SignalingRadioBearer.txt', sep='\t', index=None)

print('SignalingRadioBearer OK')

In [ ]:
import pandas as pd
from glob import glob
import csv
import numpy as np

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\EUtranFrequency_*_part*.parquet")

# básico: NaN -> "LTE", não-nulo -> "NR"
df1["Tecnologia"] = np.where(df1["GNBCUCPFunction"].isna(), "LTE", "NR")
df1['earfcndl']=df1['arfcnValueEUtranDl']

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['EUtranFrequencyId']= df1['EUtranFrequencyId'].fillna('')+df1['eUtranFrequencyId'].fillna('')

df2= pd.DataFrame(df1)
df2.drop('eUtranFrequencyId', axis='columns', inplace=True)
df3= df2.dropna(thresh=100, axis='columns')
df3.to_csv('C:/Dumps/LTE_Total/EUtranFrequency.txt', sep='\t', index=None)

print('EUtranFrequency OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\CarrierAggregationFunction_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]
df1['CarrierAggregationFunctionId']= df1['CarrierAggregationFunctionId'].fillna('').astype(str)+df1['carrierAggregationFunctionId'].fillna('').astype(str)

df1.drop('carrierAggregationFunctionId', axis='columns', inplace=True)

df1= df1[df1['eNB'].str[:1].isin(['T', 'M'])]

df2= df1.dropna(thresh=100, axis='columns')


df3.to_csv('C:/Dumps/LTE_Total/CarrierAggregationFunction.txt', sep='\t', index=None)

print('CarrierAggregationFunction OK')

In [ ]:
import pandas as pd
from glob import glob
import numpy as np

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\AdmissionControl_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['AdmissionControlId']= df1['AdmissionControlId'].fillna('').astype(str)+df1['admissionControlId'].fillna('').astype(str)

df1.drop('admissionControlId', axis='columns', inplace=True)

df1['Tecnologia'] = np.where(df1['GNBDUFunction'].notna() & (df1['GNBDUFunction'] != ''), 'NR', 'LTE')

df2= df1[(df1['Tecnologia']=='LTE') & (df1['eNB'].str[:1]!='S')]
df2= df2.dropna(thresh=100, axis='columns')
df2= df2[df2['eNB'].str[:1]!= 'W']

df2.to_csv('C:/Dumps/LTE_Total/AdmissionControl.txt', sep='\t', index=None)

print('AdmissionControl OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\AnrFunctionEUtran_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]
df1['AnrFunctionEUtranId']= df1['AnrFunctionEUtranId'].fillna('').astype(str)+df1['anrFunctionEUtranId'].fillna('').astype(str)
df1.drop('anrFunctionEUtranId', axis='columns', inplace=True)

df1['Tecnologia'] = np.where(df1['GNBCUCPFunction'].notna() & (df1['GNBCUCPFunction'] != ''), 'NR', 'LTE')

df1= df1[(df1['Tecnologia']=='LTE') & (~df1['eNB'].str[:1].isin(['S', 'W']))]
df1= df1.dropna(thresh=100, axis='columns')
df1.dropna(subset=['acquirePlmnIdListEnabled'], inplace=True)

df1.to_csv('C:/Dumps/LTE_Total/AnrFunctionEUtran.txt', sep='\t', index=None)

print('AnrFunctionEUtran OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\AnrFunction_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]
df1['AnrFunctionId']= df1['AnrFunctionId'].fillna('').astype(str)+df1['anrFunctionId'].fillna('').astype(str)
df1.drop('anrFunctionId', axis='columns', inplace=True)

df1['Tecnologia'] = np.where(df1['GNBCUCPFunction'].notna() & (df1['GNBCUCPFunction'] != ''), 'NR', 'LTE')

df1= df1[(df1['Tecnologia']=='LTE') & (~df1['eNB'].str[:1].isin(['S', 'W']))]
df1= df1.dropna(thresh=100, axis='columns')

df1.dropna(subset=['cellRelHoAttRateThreshold'], inplace=True)

df1.to_csv('C:/Dumps/LTE_Total/AnrFunction.txt', sep='\t', index=None)

print('AnrFunction OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\AnrFunctionUtran_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['AnrFunctionUtranId']= df1['AnrFunctionUtranId'].fillna('').astype(str)+df1['anrFunctionUtranId'].fillna('').astype(str)
df1.drop('anrFunctionUtranId', axis='columns', inplace=True)
df1= df1.dropna(thresh=100, axis='columns')

df1= df1[df1['eNB'].str[:1]!= 'W']

df1.to_csv('C:/Dumps/LTE_Total/AnrFunctionUtran.txt', sep='\t', index=None)

print('AnrFunctionUtran OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\NbIotCell_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]
df1['CELL'] = df1['nbIotCellId'].combine_first(df1['NbIotCellId'])
df1['NbIotCellId']= df1['NbIotCellId'].fillna('')+df1['nbIotCellId'].fillna('')

df1.drop('nbIotCellId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')
#df3.columns.to_list()


df1.to_csv('C:/Dumps/LTE_Total/NbIotCell.txt', sep='\t')
df1.to_csv('C:/Vivo Reports/LTE/LTE MOs/NbIotCell.csv', encoding='ISO-8859-1', index=None)
print('NbIotCell OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\Router_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df2= df1.dropna(thresh=1, axis='columns')

df2.to_csv('C:/Dumps/LTE_Total/Router_LTE.txt', sep='\t', index=None)

print('Router_LTE OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\InterfaceIPv4_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df2= df1.dropna(thresh=1, axis='columns')

df2.to_csv('C:/Dumps/LTE_Total/InterfaceIPv4_LTE.txt', sep='\t', index=None)

print('InterfaceIPv4_LTE OK')

import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\InterfaceIPv6_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df2= df1.dropna(thresh=1, axis='columns')

df2.to_csv('C:/Dumps/LTE_Total/InterfaceIPv6_LTE.txt', sep='\t', index=None)

print('Finished')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\Paging_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['PagingId']= df1['PagingId'].fillna('').astype(str)+df1['pagingId'].fillna('').astype(str)

df2= pd.DataFrame(df1)
df2.drop('pagingId', axis='columns', inplace=True)
df2.dropna(subset=['maxNoOfPagingRecords'], inplace=True)

df2.to_csv('C:/Dumps/LTE_Total/Paging.txt', sep='\t', index=None)

print('Paging OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigA1Prim_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)
df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigA1PrimId']= df1['ReportConfigA1PrimId'].fillna('').astype(str)+df1['reportConfigA1PrimId'].fillna('').astype(str)
df1.drop('reportConfigA1PrimId', axis='columns', inplace=True)

df2= df1.dropna(thresh=100, axis='columns')

df2.to_csv('C:/Dumps/LTE_Total/ReportConfigA1Prim.txt', sep='\t', index=None)
print('ReportConfigA1Prim OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigA1Sec_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)
df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigA1SecId']= df1['ReportConfigA1SecId'].fillna('').astype(str)+df1['reportConfigA1SecId'].fillna('').astype(str)

df2= pd.DataFrame(df1)

df2.drop('reportConfigA1SecId', axis='columns', inplace=True)

df2= df2.dropna(thresh=100, axis='columns')

df2.to_csv('C:/Dumps/LTE_Total/ReportConfigA1Sec.txt', sep='\t', index=None)
print('finished!!!')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigB1Geran_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)
df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigB1GeranId']= df1['ReportConfigB1GeranId'].fillna('').astype(str)+df1['reportConfigB1GeranId'].fillna('').astype(str)

df1.drop('reportConfigB1GeranId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigB1Geran.txt', sep='\t', index=None)

print('ReportConfigB1Geran OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigB1Utra_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigB1UtraId']= df1['ReportConfigB1UtraId'].fillna('').astype(str)+df1['reportConfigB1UtraId'].fillna('').astype(str)

df1.drop('reportConfigB1UtraId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigB1Utra.txt', sep='\t', index=None)

print('ReportConfigB1Utra OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigB1GUtra_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigB1GUtraId']= df1['ReportConfigB1GUtraId'].fillna('').astype(str)+df1['reportConfigB1GUtraId'].fillna('').astype(str)
df1.drop('reportConfigB1GUtraId', axis='columns', inplace=True)
df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigB1GUtra.txt', sep='\t', index=None)
print('ReportConfigB1GUtra OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigB1NR_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigB1NRId']= df1['ReportConfigB1NRId'].fillna('').astype(str)+df1['reportConfigB1NRId'].fillna('').astype(str)

df1.drop('reportConfigB1NRId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigB1NR.txt', sep='\t', index=None)
print('ReportConfigB1NR OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigEUtraBadCovPrim_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigEUtraBadCovPrimId']= df1['ReportConfigEUtraBadCovPrimId'].fillna('').astype(str)+df1['reportConfigEUtraBadCovPrimId'].fillna('').astype(str)

df1.drop('reportConfigEUtraBadCovPrimId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigEUtraBadCovPrim.txt', sep='\t', index=None)
print('ReportConfigEUtraBadCovPrim OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigEUtraBadCovSec_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]
df1['ReportConfigEUtraBadCovSecId']= df1['ReportConfigEUtraBadCovSecId'].fillna('').astype(str)+df1['reportConfigEUtraBadCovSecId'].fillna('').astype(str)
df1.drop('reportConfigEUtraBadCovSecId', axis='columns', inplace=True)

df1.dropna(subset=['CELL'], inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigEUtraBadCovSec.txt', sep='\t', index=None)
print('ReportConfigEUtraBadCovSec OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigEUtraBestCell_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigEUtraBestCellId']= df1['ReportConfigEUtraBestCellId'].fillna('').astype(str)+df1['reportConfigEUtraBestCellId'].fillna('').astype(str)

df1.drop('reportConfigEUtraBestCellId', axis='columns', inplace=True)
df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigEUtraBestCell.txt', sep='\t', index=None)
print('ReportConfigEUtraBestCell OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigA5UlTraffic_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigA5UlTrafficId']= df1['ReportConfigA5UlTrafficId'].fillna('').astype(str)+df1['reportConfigA5UlTrafficId'].fillna('').astype(str)

df1.drop('reportConfigA5UlTrafficId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigA5UlTraffic.txt', sep='\t', index= None)
print('ReportConfigA5UlTraffic OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\ReportConfigSCellA1A2_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['Banda']= df1['CELL'].str[0:1].replace(['T','Q', 'V', 'Z', 'U', 'P', 'Y', 'C', 'L', 'O'],
                                           ['2600','2600', '1800', '700', '2100', '2600', '850', '1800', '2380', '2360'])

df1['UF']= df1['eNB'].str[-2:]

df1['ReportConfigSCellA1A2Id']= df1['ReportConfigSCellA1A2Id'].fillna('').astype(str)+df1['reportConfigSCellA1A2Id'].fillna('').astype(str)

df1.drop('reportConfigSCellA1A2Id', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/ReportConfigSCellA1A2.txt', sep='\t', index=None)
print('ReportConfigSCellA1A2 OK')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\DrxProfile_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['DrxProfileId']= df1['DrxProfileId'].fillna('').astype(str)+df1['drxProfileId'].fillna('').astype(str)

df1.drop('drxProfileId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/DrxProfile.txt', sep='\t', index=None)
print('DrxProfile OK')

In [ ]:
import pandas as pd
from glob import glob
import numpy as np

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\Rrc_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])

df1['Tecnologia'] = np.where(df1['GNBCUCPFunction'].notna() & (df1['GNBCUCPFunction'] != ''), 'NR', 'LTE')
df1= df1[df1['Tecnologia']=='LTE']

df1['UF']= df1['eNB'].str[-2:]
df1['RrcId']= df1['RrcId'].fillna('').astype(str)+df1['rrcId'].fillna('').astype(str)
df1.drop('rrcId', axis='columns', inplace=True)
df1.dropna(subset='ENodeBFunction', inplace=True) 
df1= df1.dropna(thresh=10, axis='columns')
df1.to_csv('C:/Dumps/LTE_Total/Rrc.txt', sep='\t', index=None)
print('Rrc OK')

In [ ]:
import pandas as pd
from glob import glob

df1 = load_parquet(fr"{base_path}\UeMeasControl_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)
df1['UF']= df1['eNB'].str[-2:]

df1['UeMeasControlId']= df1['UeMeasControlId'].fillna('').astype(str)+df1['ueMeasControlId'].fillna('').astype(str)
df1.drop('ueMeasControlId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.dropna(subset=['CELL', 'a3MobilityTimerBr'], inplace=True)

df1= df1.loc[df3['eNB'].str[:1].isin(['T', 'M']), :]

df1.to_csv('C:/Dumps/LTE_Total/UeMeasControl.txt', sep='\t', index=None)
print('UeMeasControl OK')

In [ ]:
import pandas as pd
from glob import glob

df1 = load_parquet(fr"{base_path}\Rcs_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1['RcsId']= df1['RcsId'].fillna('').astype(str)+df1['rcsId'].fillna('').astype(str)
df1.drop('rcsId', axis='columns', inplace=True)

df1= df1.dropna(thresh=100, axis='columns')

df1.to_csv('C:/Dumps/LTE_Total/Rcs.txt', sep='\t', index=None)
print('Rcs OK')

In [ ]:
import pandas as pd
from glob import glob

df1 = load_parquet(fr"{base_path}\ManagedElement_*_part*.parquet")

df1= df1.dropna(thresh=100, axis='columns')
df1['ManagedElementId']= df1['ManagedElementId'].fillna('').astype(str)+df1['managedElementId'].fillna('').astype(str)
df1.drop('managedElementId', axis='columns', inplace=True)
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['ManagedElementId'].str[-2:]

df1.to_csv('C:/Dumps/LTE_Total/ManagedElement.txt', sep='\t', index=None)
print('ManagedElement OK')

# Adicionando o MO ExternalEUtranCellFDD para audit de SP

pd.set_option('display.max_columns',None)
df1.sample(5)
teste= df1[~df1['pciConflict'].isna()]
teste1= teste[teste['CELL']=='VYTD12']
teste1

import pandas as pd
from glob import glob

filenames = glob('C:/Dumps/LTE/Export_ExternalEUtranCellFDD_*.txt')

df1= pd.DataFrame()

for f in filenames:
    df= pd.read_csv(f,delimiter='\t', low_memory=False)
    df1=pd.concat([df1,df],sort=False, axis=0)
    

df1['eNB']=df1['MO'].str.extract('MeContext=([^,]+)', expand=False)
df1['eNB'] = df1['eNB'].fillna(df1['MO'].str.extract('ManagedElement=([^,]+)', expand=False))

df1['UF']= df1['eNB'].str[-2:]

# criando a coluna CELL

df1['CELL']=df1['reservedBy'].str.extract("(EUtranCell.*\w+)",expand=True)
df1['CELL']=df1['CELL'].str[14:20]

# criando a coluna EUtranFrequency
df1['EUtranFrequency']=df1['eutranFrequencyRef'].str.extract("(EUtranFrequency=\w+)",expand=True)
df1['EUtranFrequency']=df1['EUtranFrequency'].str.replace('EUtranFrequency=',"")
df1['Freq_tgt']= df1['EUtranFrequency'].str.replace('LTE_', '')

# Criar uma nova coluna que é a combinação das duas colunas existentes
df1['ExternalEUtranCellFDDId'] = df1['externalEUtranCellFDDId'].combine_first(df1['ExternalEUtranCellFDDId'])

df1.drop(columns=['externalEUtranCellFDDId'], inplace=True)

df1= df1.dropna(thresh=10, axis='columns')

# Ordem desejada das colunas a partir da terceira coluna
colunas_desejadas = ['UF', 'eNB', 'CELL', 'EUtranFrequency', 'Freq_tgt']

# As primeiras duas colunas
primeiras_colunas = df1.columns.tolist()[:2]

# Todas as outras colunas, excluindo as colunas desejadas e as primeiras duas colunas
outras_colunas = [col for col in df1.columns if col not in colunas_desejadas + primeiras_colunas]

# Combinando as listas para obter a nova ordem das colunas
nova_ordem_colunas = primeiras_colunas + colunas_desejadas + outras_colunas

# Reordenando as colunas
df1 = df1[nova_ordem_colunas]

df1.to_csv('C:/Dumps/LTE_Total/ExternalEUtranCellFDD.txt', sep='\t')
print('Done')

# Adicionando Novos MOs para audit de Baseline

### MO AnrFunctionGeran

import pandas as pd
from glob import glob

filenames = glob('C:/Dumps/LTE/Export_AnrFunctionGeran_*.txt')

df1= pd.DataFrame()

for f in filenames:
    df= pd.read_csv(f,delimiter='\t', engine='python')
    df1=pd.concat([df1,df],sort=False, axis=0)

df1['eNB']=df1['MO'].str.extract('MeContext=([^,]+)', expand=False)
df1['eNB'] = df1['eNB'].fillna(df1['MO'].str.extract('ManagedElement=([^,]+)', expand=False))

df1['UF']= df1['eNB'].str[-2:]
df1['AnrFunctionGeranId']= df1['AnrFunctionGeranId'].fillna('').astype(str)+df1['anrFunctionGeranId'].fillna('').astype(str)
df1.drop('anrFunctionGeranId', axis='columns', inplace=True)
df1= df1[~df1['eNB'].str[:1].isin(['W', 'S'])]
df1['eNB'].str[0].unique()

df1= df1.dropna(thresh=100, axis='columns')

# Lista de todas as colunas em df1
columns = list(df1.columns)

# Removendo 'UF', 'eNB', 'CELL' da lista original para evitar duplicatas
columns = [col for col in columns if col not in ['UF', 'eNB']]

# Inserindo 'UF', 'eNB', 'CELL' a partir da terceira posição
new_columns = columns[:2] + ['UF', 'eNB'] + columns[2:]

# Reorganizando df1 com a nova ordem de colunas
df1 = df1[new_columns]

df1.to_csv('C:/Dumps/LTE_Total/AnrFunctionGeran.txt', sep='\t')

print('Done')

# MO AddressIPv4 - Usado para fornecer o IP dos termpoints

In [ ]:
import pandas as pd
from glob import glob

df1 = load_parquet(fr"{base_path}\AddressIPv4_*_part*.parquet")

df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['eNB'] = df1['eNB'].astype("string")          # necessário p/ .str
df1['UF'] = df1['eNB'].str[-2:]

df1['IP_address'] = df1['usedAddress'].astype("string").str.extract(r'(?P<IP>[^/]+)', expand=True)  # necessário p/ ArrowDtype
df1.dropna(thresh=1, axis=1, inplace=True)

columns = list(df1.columns)
columns = [col for col in columns if col not in ['UF', 'eNB', 'IP_address']]
new_columns = columns[:2] + ['UF', 'eNB', 'IP_address'] + columns[2:]
df1 = df1[new_columns]

df1.to_csv('C:/Dumps/LTE_Total/AddressIPv4.txt', sep='\t', index=None)
df1.to_csv('C:/Vivo Reports/5G/5G MOs/AddressIPv4.csv', index=None)

print('AddressIPv4, Done')

In [ ]:
import pandas as pd
from glob import glob

# Carrega TODOS os parts do MO 
df1 = load_parquet(fr"{base_path}\Slot_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1['UF']= df1['eNB'].str[-2:]

df1.dropna(subset='productData_productName',inplace=True)
df1.dropna(thresh=1, axis='columns', inplace=True)
df1.to_csv('C:/Dumps/LTE_Total/Slot.txt', sep='\t')
df1.to_csv('C:/Vivo Reports/MoB_IMP/Slot.txt', sep='\t', index=None)
print('Slot OK')

In [ ]:
import pandas as pd
from glob import glob

df1 = load_parquet(fr"{base_path}\CellSleepFunction_*_part*.parquet")
df1['eNB'] = df1['eNB'].combine_first(df1['ManagedElement'])
df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

df1['UF']= df1['eNB'].str[-2:]

df1= df1.dropna(thresh=1, axis='columns')

# df1= df1.set_index('FileName')
df1['Regional']= df1['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

# Lista com as colunas que você quer mover
cols_to_move = ['Regional', 'UF', 'eNB', 'CELL']

# Todas as colunas do df1
all_cols = df1.columns.tolist()

# Mantém as duas primeiras colunas
first_two = all_cols[:2]

# Remove as colunas a mover da lista original
remaining_cols = [col for col in all_cols if col not in cols_to_move and col not in first_two]

# Nova ordem: primeiras duas colunas, depois as colunas a mover, depois o resto
new_order = first_two + cols_to_move + remaining_cols

# Reordena o dataframe
df1 = df1[new_order]

df1.to_csv('C:/Vivo Reports/LTE/LTE MOs/CellSleepFunction.csv', index=None)

print('CellSleepFunction OK')

In [ ]:
# Adicionando o ExternalENodeBFunction para o audit Missing_EUtranCellRelation_5G_4G

import pandas as pd


df1 = load_parquet(fr"{base_path}\ExternalENodeBFunction_*_part*.parquet")
df1['eNB'] = df1['ManagedElement']

df1.rename(columns={'EUtranCell': 'CELL'}, inplace=True)

# Filtrando apenas as linhas 5G
df1= df1[~df1['GNBCUCPFunction'].isna()]
df1= df1.dropna(thresh=1, axis='columns')
df1.to_csv('C:/Camilo/5G_ExternalENodeBFunction.txt', sep='\t', index=None)
print('5G_ExternalENodeBFunction OK')

# MO UtranCellRelation 4G--3G

import pandas as pd
from glob import glob

filenames = glob('C:/Dumps/LTE/Export_UtranCellRelation_*.txt')

df1= pd.DataFrame()

for f in filenames:
    df= pd.read_csv(f,delimiter='\t', low_memory=False)
    df1=pd.concat([df1,df],sort=False, axis=0)
    

df1['eNB']=df1['MO'].str.extract('MeContext=([^,]+)', expand=False)
df1['eNB'] = df1['eNB'].fillna(df1['MO'].str.extract('ManagedElement=([^,]+)', expand=False))

df1['UF']= df1['eNB'].str[-2:]
pd.set_option('display.max_columns',39)
pd.set_option('display.max_colwidth', None)

# criando a coluna CELL

df1['CELL']=df1['MO'].str.extract("(EUtranCell.*\w+)",expand=True)
df1['CELL']=df1['CELL'].str[14:20]
df1['UtranCellRelationId'] = df1['utranCellRelationId'].combine_first(df1['UtranCellRelationId'])
df1.head()

# df2= df1[df1['UF']=='SP']
df2.to_excel('C:/Camilao/UtranCellRelation_4G_3G.xlsx',index=None)

df2= df1.copy()

df2=df2[['UF', 'eNB', 'CELL', 'UtranCellRelationId']]

df2.head()

eutrancell= pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.csv', encoding='ISO-8859-1', usecols=['UF', 'MUNICIPIO', 'eNB', 'CELL', 'operationalState'])
eutrancell.rename(columns={'UF': 'UF_src', 'eNB': 'eNB_src', 'CELL': 'CELL_src'}, inplace=True)
eutrancell.head()

audit= pd.merge(eutrancell, df2, left_on= ['UF_src', 'eNB_src', 'CELL_src'], right_on= ['UF', 'eNB', 'CELL'], how='left', suffixes=['_src', '_tgt'])

audit1= audit[audit['eNB'].isna()]   

audit1.to_excel('C:/Camilao/Missing_UtranCellRelation_4G_3G.xlsx', index=None)
print('Done')